# Policy Analysis

Load a trained checkpoint, run inference on all levels, cache logits to disk,
and analyze per-level win probabilities to understand the hard tail.

In [ ]:
import sys
from pathlib import Path

# Add project root to path so `src.*` imports work from notebooks/
sys.path.insert(0, str(Path("..").resolve()))

import json

import equinox as eqx
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import mummymaze_rust
import numpy as np

from src.train.checkpoint import load_checkpoint
from src.train.dataset import load_bc_dataset, make_batch_obs
from src.train.wire import next_power_of_2

MAZE_DIR = Path("../mazes")
CHECKPOINT_DIR = Path("../checkpoints/resnet-attn-width4/epoch050")
CACHE_PATH = Path("../cache/policy_analysis_width4_epoch050.npz")

## 1. Load model and dataset

In [ ]:
ckpt = load_checkpoint(CHECKPOINT_DIR)
model = ckpt.model
print(f"Loaded: arch={ckpt.arch}, hparams={ckpt.hparams}, epoch={ckpt.epoch}")
n_params = sum(x.size for x in jax.tree.leaves(eqx.filter(model, eqx.is_array)))
print(f"Parameters: {n_params:,}")

datasets, sources = load_bc_dataset(MAZE_DIR)
for gs, ds in sorted(datasets.items()):
    n_train = int(ds.train_mask.sum())
    n_val = int(ds.val_mask.sum())
    n_levels = len(sources[gs])
    print(f"  gs={gs}: {ds.n_states} states, {n_levels} levels ({n_train} train, {n_val} val)")

## 2. Run inference on all states, cache logits to disk

Computes logits for every state (train + val) across all grid sizes.
Saves to an `.npz` file so we never need to re-run inference.

In [ ]:
BATCH_SIZE = 1024

if CACHE_PATH.exists():
    print(f"Loading cached logits from {CACHE_PATH}")
    cached = np.load(CACHE_PATH, allow_pickle=True)
    all_logits = {int(k.split("_")[1]): cached[k] for k in cached if k.startswith("logits_")}
    print(f"  Loaded: {', '.join(f'gs={gs}: {v.shape}' for gs, v in sorted(all_logits.items()))}")
else:
    print("Running inference on all states (this may take a few minutes)...")
    CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)

    # Pre-JIT obs construction and forward pass per grid size
    jit_make_obs = {}
    jit_forward = {}
    for gs, ds in datasets.items():
        jit_make_obs[gs] = jax.jit(
            lambda tuples, lidx, gs=gs, bank=ds.bank: make_batch_obs(gs, bank, tuples, lidx)
        )
        jit_forward[gs] = jax.jit(jax.vmap(model))
        # Warm up both
        dummy_obs = jit_make_obs[gs](ds.state_tuples[:1], ds.level_idx[:1])
        _ = jit_forward[gs](dummy_obs)

    all_logits = {}
    for gs, ds in sorted(datasets.items()):
        n = ds.n_states
        logits_list = []
        for start in range(0, n, BATCH_SIZE):
            end = min(start + BATCH_SIZE, n)
            obs = jit_make_obs[gs](ds.state_tuples[start:end], ds.level_idx[start:end])
            logits_list.append(np.array(jit_forward[gs](obs)))
            if start % (BATCH_SIZE * 100) == 0:
                print(f"  gs={gs}: {start}/{n}")
        all_logits[gs] = np.concatenate(logits_list, axis=0)
        print(f"  gs={gs}: done — {all_logits[gs].shape}")

    # Save to disk
    save_dict = {f"logits_{gs}": logits for gs, logits in all_logits.items()}
    np.savez(CACHE_PATH, **save_dict)
    print(f"Saved logits to {CACHE_PATH}")

## 3. Compute per-level win probabilities via Markov solver

Uses cached logits → softmax → Rust Markov solver for exact win probability per level.

In [ ]:
def softmax(logits):
    """Numerically stable softmax."""
    x = logits - logits.max(axis=-1, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=-1, keepdims=True)


def parse_rust_levels(maze_dir, keys):
    """Parse Rust Level objects from (stem, sub) keys."""
    by_stem = {}
    for stem, sub in keys:
        by_stem.setdefault(stem, []).append(sub)

    stem_levels = {}
    for stem, subs in by_stem.items():
        path = maze_dir / f"{stem}.dat"
        all_levels = mummymaze_rust.parse_file(str(path))
        for sub in subs:
            stem_levels[(stem, sub)] = all_levels[sub]

    return [stem_levels[k] for k in keys]


# Compute win_prob for ALL levels (train + val), skipping empty levels
level_results = []

for gs, ds in sorted(datasets.items()):
    logits = all_logits[gs]
    probs = softmax(logits)

    level_idx_np = np.array(ds.level_idx)
    level_keys = list(sources[gs])
    n_levels = len(level_keys)

    counts = np.bincount(level_idx_np, minlength=n_levels)

    # Only process levels that have states in the dataset
    active_indices = [i for i in range(n_levels) if counts[i] > 0]
    if not active_indices:
        continue

    active_keys = [level_keys[i] for i in active_indices]
    old_to_new = {old: new for new, old in enumerate(active_indices)}

    # Remap level indices to dense range
    active_set = set(active_indices)
    mask = np.array([int(x) in active_set for x in level_idx_np])
    remapped_idx = np.array([old_to_new[int(x)] for x in level_idx_np[mask]])

    n_active = len(active_indices)
    active_counts = np.bincount(remapped_idx, minlength=n_active)
    offsets = np.zeros(n_active + 1, dtype=np.intp)
    np.cumsum(active_counts, out=offsets[1:])

    # Sort states by remapped level index
    sort_order = np.argsort(remapped_idx, kind="stable")
    sorted_state_tuples = np.asarray(ds.state_tuples, dtype=np.int32)[mask][sort_order]
    sorted_probs = probs[mask][sort_order]

    rust_levels = parse_rust_levels(MAZE_DIR, active_keys)

    win_probs = mummymaze_rust.policy_win_prob_batch(
        rust_levels, sorted_state_tuples, sorted_probs, offsets.tolist()
    )

    for j, orig_idx in enumerate(active_indices):
        stem, sub = level_keys[orig_idx]
        wp = win_probs[j]
        a = mummymaze_rust.analyze(rust_levels[j])
        is_val = bool(np.any(np.array(ds.val_mask)[level_idx_np == orig_idx]))

        level_results.append({
            "grid_size": gs,
            "stem": stem,
            "sub": sub,
            "key": f"{stem}:{sub}",
            "is_val": is_val,
            "n_states": int(counts[orig_idx]),
            "win_prob": wp,
            "bfs_moves": a["bfs_moves"],
            "bfs_win_prob": a["win_prob"],
            "expected_steps": a["expected_steps"],
        })

    active_wps = np.array(win_probs)
    print(f"gs={gs}: {n_active} levels (of {n_levels}), "
          f"mean_wp={np.nanmean(active_wps):.4f}, "
          f"median_wp={np.nanmedian(active_wps):.4f}")

print(f"\nTotal: {len(level_results)} levels (with states)")

## 4. Win probability distribution and hard tail analysis

In [ ]:
# Split into train/val
val_results = [r for r in level_results if r["is_val"]]
train_results = [r for r in level_results if not r["is_val"]]

val_wps = np.array([r["win_prob"] for r in val_results])
train_wps = np.array([r["win_prob"] for r in train_results])

print(f"Train: mean={np.nanmean(train_wps):.4f}, median={np.nanmedian(train_wps):.4f}")
print(f"Val:   mean={np.nanmean(val_wps):.4f}, median={np.nanmedian(val_wps):.4f}")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Overall distribution
ax = axes[0]
ax.hist(val_wps[~np.isnan(val_wps)], bins=50, alpha=0.6, label="val", density=True)
ax.hist(train_wps[~np.isnan(train_wps)], bins=50, alpha=0.4, label="train", density=True)
ax.set_xlabel("Win Probability")
ax.set_ylabel("Density")
ax.set_title("Win Probability Distribution")
ax.legend()
ax.grid(True, alpha=0.3)

# By grid size
ax = axes[1]
for gs in [6, 8, 10]:
    wps = np.array([r["win_prob"] for r in val_results if r["grid_size"] == gs])
    wps = wps[~np.isnan(wps)]
    ax.hist(wps, bins=30, alpha=0.5, label=f"gs={gs}", density=True)
    print(f"  val gs={gs}: mean={wps.mean():.4f}, median={np.median(wps):.4f}, "
          f"<50%: {(wps < 0.5).sum()}/{len(wps)}")
ax.set_xlabel("Win Probability")
ax.set_title("Val Win Prob by Grid Size")
ax.legend()
ax.grid(True, alpha=0.3)

# Hard tail: levels with win_prob < 50%
ax = axes[2]
hard = [r for r in val_results if r["win_prob"] < 0.5 and not np.isnan(r["win_prob"])]
hard_wps = np.array([r["win_prob"] for r in hard])
if len(hard_wps) > 0:
    ax.hist(hard_wps, bins=30, alpha=0.7, color="tab:red")
ax.set_xlabel("Win Probability")
ax.set_title(f"Hard Tail: {len(hard)} val levels with win_prob < 50%")
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. What makes levels hard?

Correlate win_prob with level features: BFS solution length, state count, grid size, BFS win probability (optimal random-walk baseline).

In [ ]:
# Scatter plots: win_prob vs level features
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

features = [
    ("bfs_moves", "BFS Solution Length"),
    ("n_states", "Number of States"),
    ("bfs_win_prob", "BFS Win Prob (optimal)"),
    ("expected_steps", "Expected Steps (optimal)"),
    ("grid_size", "Grid Size"),
]

for ax, (feat, title) in zip(axes.flatten(), features):
    for split, results, color, alpha in [
        ("val", val_results, "tab:blue", 0.5),
        ("train", train_results, "tab:gray", 0.1),
    ]:
        xs = [r[feat] for r in results if not np.isnan(r["win_prob"])]
        ys = [r["win_prob"] for r in results if not np.isnan(r["win_prob"])]
        ax.scatter(xs, ys, s=3, alpha=alpha, label=split, color=color)
    ax.set_xlabel(title)
    ax.set_ylabel("Agent Win Prob")
    ax.set_title(f"Win Prob vs {title}")
    ax.legend(markerscale=4)
    ax.grid(True, alpha=0.3)

# Correlation table
ax = axes[1, 2]
ax.axis("off")
val_wp = np.array([r["win_prob"] for r in val_results if not np.isnan(r["win_prob"])])
rows = []
for feat, title in features:
    vals = np.array([r[feat] for r in val_results if not np.isnan(r["win_prob"])])
    if vals.dtype in (np.float64, np.float32, np.int64, np.int32):
        corr = np.corrcoef(vals, val_wp)[0, 1]
        rows.append([title, f"{corr:.3f}"])
table = ax.table(cellText=rows, colLabels=["Feature", "Corr w/ win_prob"],
                 loc="center", cellLoc="left")
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 1.5)
ax.set_title("Correlation with Win Prob (val)")

plt.tight_layout()
plt.show()

## 6. Hardest levels

Bottom 50 levels by win probability — what do they have in common?

In [ ]:
# Sort all levels by win_prob (ascending = hardest first)
sorted_results = sorted(
    [r for r in level_results if not np.isnan(r["win_prob"])],
    key=lambda r: r["win_prob"],
)

# Show hardest 50
print(f"{'Rank':>4} {'Key':>12} {'GS':>3} {'Split':>5} {'WinProb':>8} "
      f"{'BFS':>4} {'States':>6} {'BFS_WP':>7}")
print("-" * 60)
for i, r in enumerate(sorted_results[:50]):
    split = "val" if r["is_val"] else "train"
    bfs = f"{r['bfs_moves']:4d}" if r["bfs_moves"] is not None else "   -"
    bfs_wp = f"{r['bfs_win_prob']:7.4f}" if r["bfs_win_prob"] is not None else "      -"
    print(f"{i+1:4d} {r['key']:>12} {r['grid_size']:3d} {split:>5} "
          f"{r['win_prob']:8.4f} {bfs} {r['n_states']:6d} {bfs_wp}")

# Summary stats for hard tail
print(f"\n--- Hard tail summary (win_prob < 0.5) ---")
hard_all = [r for r in sorted_results if r["win_prob"] < 0.5]
print(f"Count: {len(hard_all)}")
if hard_all:
    gs_counts = {}
    for r in hard_all:
        gs_counts[r["grid_size"]] = gs_counts.get(r["grid_size"], 0) + 1
    print(f"By grid size: {dict(sorted(gs_counts.items()))}")
    bfs_moves = [r["bfs_moves"] for r in hard_all if r["bfs_moves"] is not None]
    if bfs_moves:
        print(f"Mean BFS moves: {np.mean(bfs_moves):.1f}")
    print(f"Mean n_states: {np.mean([r['n_states'] for r in hard_all]):.1f}")
    bfs_wps = [r["bfs_win_prob"] for r in hard_all if r["bfs_win_prob"] is not None]
    if bfs_wps:
        print(f"Mean BFS win_prob: {np.mean(bfs_wps):.4f}")

## 7. Train vs val generalization gap

Does the model do better on train levels than val levels? This shows overfitting at the level (not state) granularity.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, gs in zip(axes, [6, 8, 10]):
    train_wp = np.array([r["win_prob"] for r in train_results
                         if r["grid_size"] == gs and not np.isnan(r["win_prob"])])
    val_wp = np.array([r["win_prob"] for r in val_results
                       if r["grid_size"] == gs and not np.isnan(r["win_prob"])])

    ax.hist(train_wp, bins=30, alpha=0.5, label=f"train (mean={train_wp.mean():.3f})", density=True)
    ax.hist(val_wp, bins=30, alpha=0.5, label=f"val (mean={val_wp.mean():.3f})", density=True)
    ax.set_xlabel("Win Probability")
    ax.set_ylabel("Density")
    ax.set_title(f"Grid Size {gs}")
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle("Train vs Val Win Probability Distribution", fontsize=14)
plt.tight_layout()
plt.show()

# Summary table
print(f"{'GS':>3} {'Train Mean':>11} {'Val Mean':>11} {'Gap':>7} {'Train <50%':>11} {'Val <50%':>11}")
print("-" * 60)
for gs in [6, 8, 10]:
    tw = np.array([r["win_prob"] for r in train_results
                   if r["grid_size"] == gs and not np.isnan(r["win_prob"])])
    vw = np.array([r["win_prob"] for r in val_results
                   if r["grid_size"] == gs and not np.isnan(r["win_prob"])])
    print(f"{gs:3d} {tw.mean():11.4f} {vw.mean():11.4f} {tw.mean() - vw.mean():7.4f} "
          f"{(tw < 0.5).sum():>5}/{len(tw):<5} {(vw < 0.5).sum():>5}/{len(vw):<5}")